# Sprint 1 — Data Pipeline End-to-End Check

Verifies the leakage-safe pipeline: Football-Data raw → canonical → `(X, y, meta)` feature matrix.
Run the cells top-to-bottom. The key one is **Cell 2** (NaN check) — X must be fully dense before it goes into a `RandomForestClassifier`.

### Cell 1 — build the matrix

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd

# Make sure the project root is importable (this notebook lives in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features.build_features import build_features

# Don't overwrite the committed schema artifact from a notebook run
X, y, meta = build_features(save_schema=False)
print(f"X: {X.shape}   y: {y.shape}   meta: {meta.shape}")
X.head()

### Cell 2 — shapes, dtypes, and the all-important NaN check

In [ ]:
print("Feature columns:", list(X.columns))
print("\nDtypes:\n", X.dtypes)

nan_counts = X.isna().sum()
print("\nColumns with NaNs (must be empty for sklearn):")
print(nan_counts[nan_counts > 0] if nan_counts.any() else "✅ None — X is fully dense")

### Cell 3 — label distribution (sanity-check class balance)

In [ ]:
print("y label counts:\n", y.value_counts())
print("\ny proportions:\n", (y.value_counts(normalize=True).round(3)))
# Expect home-win the largest class (home advantage), draws smallest-ish.

### Cell 4 — alignment + temporal ordering (the leakage-relevant check)

In [ ]:
assert len(X) == len(y) == len(meta), "X/y/meta length mismatch"
assert list(X.index) == list(meta.index) == list(y.index), "indices not aligned"

meta = meta.copy()
meta["date"] = pd.to_datetime(meta["date"])
print("Date range:", meta["date"].min().date(), "→", meta["date"].max().date())
print("Seasons:", sorted(meta['season'].unique()))
print("Rows monotonic in date:", meta["date"].is_monotonic_increasing)
# If not monotonic, a time-ordered sort is needed before a walk-forward split (Sprint 2).

### Cell 5 — leakage spot-check on the earliest match

In [ ]:
# The very first match in the data can have NO prior history.
# Its rolling features should be imputed/neutral, never derived from later games.
first = meta.sort_values("date").iloc[0]
row = X.loc[meta["match_id"] == first["match_id"]].iloc[0]
print(f"Earliest match: {first['home_team']} vs {first['away_team']} ({first['date'].date()})")
print(row[[c for c in X.columns if "rolling" in c or "elo" in c or "position" in c]])
# Elo should sit near the 1450 start; rolling rates near their imputed neutral values.

### Cell 6 — a real temporal split (how Sprint 2 will train) + the validation artifacts

In [ ]:
# Train on all-but-last season, "predict" the last — proves the matrix supports a clean time split.
last_season = sorted(meta["season"].unique())[-1]
train_idx = meta.index[meta["season"] < last_season]
test_idx  = meta.index[meta["season"] == last_season]
print(f"Train: {len(train_idx)} matches (< {last_season})   Test: {len(test_idx)} matches ({last_season})")
print("Max train date < min test date:",
      meta.loc[train_idx, "date"].max() < meta.loc[test_idx, "date"].min())

# Show the data-quality artifacts the ingest step produced
report = json.loads((ROOT / "data/processed/validation_report.json").read_text())
print("\nValidation report:", json.dumps({k: report[k] for k in
      ["total_rows","rows_kept","rows_dropped","dropped_by_reason"]}, indent=2))
schema = json.loads((ROOT / "artifacts/feature_schema.json").read_text())
print("Locked schema n_features:", schema["n_features"])